<a href="https://colab.research.google.com/github/2403a52261-jpg/NLP/blob/main/NLP_Assignment_14_4_2403a52261.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split


texts = [
    "free money now",
    "call now free offer",
    "hello how are you",
    "let us meet tomorrow",
    "win cash prize",
    "are you available tomorrow"
]

labels = [1, 1, 0, 0, 1, 0]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts).toarray()
y = labels


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=2)


class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)


        flattened_size = kernels * ((input_size - pool_size)//2)


        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)


        output_neurons=1
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return self.sigmoid(x)


model = CNNModel(input_size=X.shape[1])


criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze()
        preds = (output > 0.5).int()
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())

y_actual = [int(y) for y in y_actual]
print(y_pred)

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Epoch 1, Loss: 0.6801
Epoch 2, Loss: 0.6801
Epoch 3, Loss: 0.6645
Epoch 4, Loss: 0.6460
Epoch 5, Loss: 0.6284
Epoch 6, Loss: 0.6104
Epoch 7, Loss: 0.5672
Epoch 8, Loss: 0.5801
Epoch 9, Loss: 0.5686
Epoch 10, Loss: 0.4831
[1, 0]

Confusion Matrix:
[[1 0]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Accuracy Score: 1.0
